In [1]:
from jax import random, numpy as jnp
from jax.scipy.stats import norm

In [22]:
def neg_LL(pred, loc, precision):
    sigma = jnp.sqrt(1 / precision)
    return -jnp.sum(norm.logpdf(pred, loc=loc, scale=sigma))


def normal_LL(x, mu, tau):
    # tau = 1 / sigma**2, for numerical reasons.
    n_samples = x.shape[0]

    mse = 1 / 2 * jnp.mean((x - mu) ** 2, axis=0)
    # 2 before tau to compensate for 1/2
    p = -n_samples / 2 * (2 * tau * mse - jnp.log(tau) + jnp.log(2 * jnp.pi))
    return p, mse

In [23]:
key = random.PRNGKey(42)
key_loc, _ = random.split(key)
x = random.normal(key, (10, ))
loc = random.normal(key_loc, x.shape)

In [24]:
neg_LL(x, loc, 100)

DeviceArray(433.10156, dtype=float32)

In [25]:
normal_LL(x, loc, 100)

(DeviceArray(-433.10153, dtype=float32),
 DeviceArray(0.44693804, dtype=float32))

In [2]:
a = jnp.ones((1, 2))

In [3]:
b, c = a

ValueError: not enough values to unpack (expected 2, got 1)

In [4]:
a

DeviceArray([[1., 1.]], dtype=float32)

In [6]:
from flax import linen as nn

In [53]:
class BiasAdderWithRunningMean(nn.Module):
  decay: float = 0.99

  @nn.compact
  def __call__(self, x):
    # easy pattern to detect if we're initializing via empty variable tree
    is_initialized = self.has_variable('batch_stats', 'mean')
    ra_mean = self.variable('batch_stats', 'mean',
                            lambda s: jnp.zeros(s),
                            x.shape[1:])
    mean = ra_mean.value # This will get either the value, or trigger init
    bias = self.param('bias', lambda rng, shape: jnp.zeros(shape), x.shape[1:])
    if is_initialized:
        ra_mean.value = self.decay * ra_mean.value + (1.0 - self.decay) * jnp.mean(x, axis=0, keepdims=True)
    return x - ra_mean.value + bias


key1, key2 = random.split(random.PRNGKey(0), 2)
x = jnp.ones((10,5))
model = BiasAdderWithRunningMean()
variables = model.init(key1, x)
#print('initialized variables:\n', variables)
y, updated_state = model.apply(variables, x, mutable=['batch_stats'])
#print('updated state:\n', updated_state)


In [57]:
y, updated_state = model.apply(variables, x, mutable=[])

ValueError: too many values to unpack (expected 2)

In [41]:
model.has_variable("batch_stats", "mean")

ValueError: Can't access variables on unbound modules

In [37]:
model

BiasAdderWithRunningMean(
    # attributes
    decay = 0.99
)

In [32]:
variables['batch_stats']['mean'] = 1.0

ValueError: FrozenDict is immutable.

In [52]:
model.variable('batch_stats', 'mean')

TypeError: variable() missing 1 required positional argument: 'init_fn'

In [58]:
a = {"aap": 1, "noot": 2}

In [61]:
a["aap", "noot"]

KeyError: ('aap', 'noot')

In [62]:
state

NameError: name 'state' is not defined

In [2]:
from modax.models import DeepmodBayesPrecalc
# %% Imports
from jax import random, numpy as jnp

from modax.data.burgers import burgers
from modax.models import DeepmodBayes, Deepmod
from modax.training import train_probabilistic, train, train_probabilistic_mse
from modax.linear_model.mask_estimator import ThresholdedLasso
from flax import optim

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

%load_ext autoreload
%autoreload 2

<frozen importlib._bootstrap>:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject
<frozen importlib._bootstrap>:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject
<frozen importlib._bootstrap>:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject


In [3]:
# %% Making data
key = random.PRNGKey(42)

x = jnp.linspace(-3, 4, 50)
t = jnp.linspace(0.5, 5.0, 20)
t_grid, x_grid = jnp.meshgrid(t, x, indexing="ij")
u = burgers(x_grid, t_grid, 0.1, 1.0)

X = jnp.concatenate([t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)], axis=1)
y = u.reshape(-1, 1)
y += 0.10 * jnp.std(y) * random.normal(key, y.shape)

In [4]:
model = DeepmodBayesPrecalc([50, 50, 1], (0.0, 0.0), (0.0, 0.0))